In [ ]:
# runs in Chameleon Jupyter environment
from chi import server, context, lease, network
import chi, os, datetime

context.version = "1.0"
context.choose_project()
context.choose_site(default="KVM@TACC")

username = os.getenv("USER")
PROJECT_SUFFIX = "proj13"
LEASE_NAME = f"lease-serve-monitor-{username}-{PROJECT_SUFFIX}"
SERVER_NAME = f"node-serve-monitor-{username}-{PROJECT_SUFFIX}"
FLAVOR_NAME = "m1.medium"
IMAGE_NAME = "CC-Ubuntu24.04"

In [ ]:
# runs in Chameleon Jupyter environment
l = lease.Lease(LEASE_NAME, duration=datetime.timedelta(hours=8))
l.add_flavor_reservation(id=chi.server.get_flavor_id(FLAVOR_NAME), amount=1)
l.submit(idempotent=True)
l.show()

In [ ]:
# runs in Chameleon Jupyter environment
s = server.Server(
    SERVER_NAME,
    image_name=IMAGE_NAME,
    flavor_name=l.get_reserved_flavors()[0].name,
)
s.submit(idempotent=True)

In [ ]:
# runs in Chameleon Jupyter environment
s.associate_floating_ip()
s.refresh()
s.check_connectivity()

In [ ]:
# runs in Chameleon Jupyter environment
security_groups = [
    {"name": "allow-ssh", "port": 22, "description": "Enable SSH traffic on TCP port 22"},
    {"name": "allow-8888", "port": 8888, "description": "Enable TCP port 8888 (Jupyter)"},
    {"name": "allow-8000", "port": 8000, "description": "Enable TCP port 8000 (serving API)"},
    {"name": "allow-9090", "port": 9090, "description": "Enable TCP port 9090 (Prometheus)"},
    {"name": "allow-3000", "port": 3000, "description": "Enable TCP port 3000 (Grafana)"},
]

for sg in security_groups:
    secgroup = network.SecurityGroup({
        "name": sg["name"],
        "description": sg["description"],
    })
    secgroup.add_rule(direction="ingress", protocol="tcp", port=sg["port"])
    secgroup.submit(idempotent=True)
    s.add_security_group(sg["name"])

print(f"updated security groups: {[sg['name'] for sg in security_groups]}")
s.refresh()
s.show(type="widget")

In [ ]:
# runs in Chameleon Jupyter environment
s.execute("curl -sSL https://get.docker.com/ | sudo sh")
s.execute("sudo groupadd -f docker; sudo usermod -aG docker $USER")
s.execute("sudo apt-get install -y docker-compose-plugin")

In [ ]:
# runs in Chameleon Jupyter environment
REPO_URL = "https://github.com/yanghao13111/SmartQueue.git"
s.execute(f"git clone {REPO_URL} ~/smartqueue")

Open an SSH session from your local terminal:

```
ssh -i ~/.ssh/id_rsa_chameleon cc@A.B.C.D
```

Replace `A.B.C.D` with the floating IP shown above.

Start the serving + monitoring stack from the VM shell:

```bash
cd ~/smartqueue/serving/docker
# Mount the trained model from training artifacts
MODEL_DIR=~/smartqueue/training/artifacts \
MLFLOW_TRACKING_URI=http://129.114.24.226:30500 \
MODEL_URI=runs:/2ce32ba692c54095b4307ae8eb7ba508/model \
docker compose -f docker-compose-monitoring.yaml up -d --build
```

Health check:

```bash
curl http://localhost:8000/health
```

Get the Jupyter token:

```bash
docker exec jupyter jupyter server list
```

Paste the token URL into a browser tab, replacing `127.0.0.1` with the VM floating IP.

Then open notebook `12_monitoring_grafana_alerts.ipynb` from the `work/` directory.

In [ ]:
# runs in Chameleon Jupyter environment
# delete the VM when finished
# s.delete()